In [ ]:
print("--- DATA ---")
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

d: list[str] = open('./data/names.txt', 'r').read().splitlines()
D: int = len(d)
v: list[str] = ['.'] + sorted(list(set(''.join(d)))) # construct vocab, with . as the start token and end token
V: int = len(v)
c2i: dict[str, int] = {c:i for i,c in enumerate(v)}
i2c: dict[int, str] = {i:c for c,i in c2i.items()}
print(f'dataset: {d[:8]} ... ({D} names)')
print(f'vocab (sample space): {v}')
print(f'I: V -> N: {c2i}')

The linear discriminant takes continuous inputs too, so the data processing of
§1.4 carries over unchanged: map (tensorize) integer indices to one-hot vectors.

The one difference is on the *output* side. Logistic regression will consume
$y^{(i)}$ as a class *index* to pluck a likelihood out of an inferred
distribution, whereas the discriminant function has no distribution to pluck
from — it regresses $K=V$ real numbers directly at the target, so the target
itself must be a vector in $\reals^{V}$. That is **1-of-K coding**: the target
for class $k$ is $\mathbf{t} = \mathbf{e}_k \in \{0,1\}^{V}$, which is just the
one-hot vector again, this time used as a regression target rather than as an
index (Bishop §5.1.3).

So both $X$ and $T$ are one-hot matrices of shape `(N, V)`, and the whole model
will be the single map $X \mapsto XW$ with $W \in \reals^{V \times V}$.

In [ ]:
xindices: list[int] = []
yindices: list[int] = []

for di in d: # loop through the dataset..
  di_normalized = ['.'] + list(di) + ['.']
  # ..in a self-supervised manner
  for x_char,y_char in zip(di_normalized, di_normalized[1:]):
    x_index, y_index = c2i[x_char], c2i[y_char] # use c2i
    xindices.append(x_index)
    yindices.append(y_index)

N = len(xindices)
xindices_N, yindices_N = torch.tensor(xindices), torch.tensor(yindices)
X_NV = F.one_hot(xindices_N, num_classes=V).float() # inputs  x^(i) in R^V
T_NV = F.one_hot(yindices_N, num_classes=V).float() # targets t^(i) in {0,1}^V, 1-of-K coded
print(f'inputs  X_NV: {X_NV.shape}')
print(f'targets T_NV: {T_NV.shape} (1-of-K, not indices)')
print(f'{D=}, {N=}, {V=}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 2))
ax1.imshow(X_NV[:5], cmap='Blues'); ax1.set_title('X_NV[:5]  (inputs)')
ax2.imshow(T_NV[:5], cmap='Oranges'); ax2.set_title('T_NV[:5]  (1-of-K targets)')
plt.show()

### Architecture: $K$ linear discriminants, one per class

A single linear discriminant $y(\mathbf{x}) = \mathbf{w}^{\top}\mathbf{x} + b$
cuts $\reals^{V}$ into the two decision regions of §1.3.2. For $K$ classes we
take $K$ of them, one per class, and decide by whichever fires hardest
(Bishop §5.1.2):

$$
y_k(\mathbf{x}) := \mathbf{w}_k^{\top}\mathbf{x} + b_k
\qquad
\hat{f}(\mathbf{x}) := \underset{k}{\argmax} \; y_k(\mathbf{x})
$$

Stacking the $\mathbf{w}_k$ as the columns of $W \in \reals^{V \times V}$ makes
the forward pass a single matmul $\mathbf{y}(\mathbf{x}) = W^{\top}\mathbf{x}$,
batched as $Y = XW$ — the same `X_NV @ W_VV` as in §1.4, except nothing
normalizes the result, so $\mathbf{y}$ is a vector of $V$ unconstrained reals
rather than a distribution. The decision is the one step; there is no inference
step to skip.

Bishop absorbs the biases by augmenting $\tilde{\mathbf{x}} = (1,
\mathbf{x})^{\top}$ and $\tilde{W} = (b, W)$. We *cannot* do that here: the
one-hot columns of $X$ already sum to the all-ones vector, so appending a
constant column makes $\tilde{X}$ rank-deficient and $\tilde{X}^{\top}\tilde{X}$
singular. The bias is already in the span of a one-hot input — one weight per
(input character, output character) pair is the whole parameterization.

In [ ]:
print("\n\n--- ARCH ---")
g = torch.Generator().manual_seed(1337+1)
W_VV = torch.randn((V, V), generator=g) # V discriminants, one per class, as columns
print(f"W_VV.shape: {W_VV.shape}, W_VV:\n{W_VV}")

def f(X_NV: torch.Tensor, W_VV: torch.Tensor) -> torch.Tensor:
  return X_NV @ W_VV                                 # y_k(x) = w_k^T x, batched. no softmax: not a distribution

def decide(Y_NV: torch.Tensor) -> torch.Tensor:
  return Y_NV.argmax(dim=1)                          # the *one* step: argmax_k y_k(x)

Y_NV = f(X_NV, W_VV)
print(f'y(x^(0)) at init: {Y_NV[0]}')
print(f'decision for {i2c[xindices_N[0].item()]!r}: {i2c[decide(Y_NV)[0].item()]!r} (target {i2c[yindices_N[0].item()]!r})')

### Training: minimizing squared error is solving a linear system

§1.3.3 scores the discriminant with the **sum-of-squares error**, each of the
$K$ outputs regressed at its 1-of-K target (Bishop §5.1.4):

$$
\mathscr{L}(W) := \tfrac{1}{2} \sum_{i=1}^{n} \| W^{\top}\mathbf{x}^{(i)} - \mathbf{t}^{(i)} \|^2
              = \tfrac{1}{2} \| XW - T \|_F^2
$$

$\mathscr{L}$ is quadratic in $W$, so setting $\nabla_W \mathscr{L} = X^{\top}(XW - T) = 0$
gives the **normal equations** — a linear system, solved in one shot rather than
iterated:

$$
(X^{\top}X)\,W = X^{\top}T
\qquad\Longrightarrow\qquad
\hat{W} = (X^{\top}X)^{-1}X^{\top}T = X^{+}T
$$

This is where §1.3 parts ways with §1.4: `argmin` is a *solve*, not a loop.

In [ ]:
print("\n\n--- TRAINING (normal equations) ---")
XtX_VV = X_NV.T @ X_NV # the Gram matrix, V x V regardless of how many examples N
XtT_VV = X_NV.T @ T_NV
W_VV = torch.linalg.solve(XtX_VV, XtT_VV) # argmin L(W) in one solve, no learning rate, no steps

loss = 0.5*((X_NV @ W_VV - T_NV)**2).sum(dim=1).mean() # per-example squared error, halved as in L(W)
print(f'W_VV.shape: {W_VV.shape}')
print(f'{loss=}')

# the same argmin, two other spellings of the same pseudo-inverse
W_lstsq_VV = torch.linalg.lstsq(X_NV, T_NV).solution # QR on X, never forms X^T X
W_pinv_VV  = torch.linalg.pinv(X_NV) @ T_NV         # X^+ T literally
print(f'||solve - lstsq||_max: {(W_VV - W_lstsq_VV).abs().max():.2e}')
print(f'||solve - pinv ||_max: {(W_VV - W_pinv_VV).abs().max():.2e}')

plt.imshow(W_VV, cmap='Purples'); plt.title('W_VV = (X^T X)^-1 X^T T'); plt.colorbar(); plt.show()

### The solved $\hat{W}$ *is* the bigram count matrix of §1.2

Look at what one-hot inputs do to the normal equations. $X^{\top}X$ counts
co-occurrences of input features with themselves — and one-hot features never
co-occur — so it is **diagonal**, holding the count of each input character.
$X^{\top}T$ counts input-character/output-character pairs, so it is exactly the
bigram count matrix $C$ of §1.2.3. Hence

$$
\hat{W} = (X^{\top}X)^{-1}X^{\top}T = \mathrm{diag}\!\left(\tfrac{1}{c_j}\right) C
$$

which is $C$ with its rows normalized: the row-stochastic matrix $P$ we
estimated by *counting*. Least-squares regression on one-hot inputs recovers the
non-parametric bigram model exactly, so the first row of the §1.3.1 table meets
the third. This is the general fact that least-squares fitted values are
class-conditional means, specialized to one-hot inputs: $\hat{W}_{jk}$ is the
mean of $t_k$ over the examples whose input is character $j$, i.e. the relative
frequency $p(y=k \mid x=j)$.

In [ ]:
print("\n\n--- TRAINING (equivalence with counting) ---")
counts_V = X_NV.sum(dim=0) # how often each character appears as an input
print(f'X^T X diagonal == input counts: {torch.equal(XtX_VV.diag(), counts_V)}')
print(f'X^T X off-diagonal max: {(XtX_VV - torch.diag(XtX_VV.diag())).abs().max()}') # one-hots never co-occur

C_VV = XtT_VV                                  # X^T T is the bigram count matrix C of 1.2.3
P_VV = C_VV / C_VV.sum(dim=1, keepdim=True)    # row-normalized: the stochastic matrix P of 1.2.3
print(f'W_hat == row-normalized counts: {torch.allclose(W_VV, P_VV, atol=1e-6)}')
print(f'||W_hat - P||_max: {(W_VV - P_VV).abs().max():.2e}')

### The same argmin by gradient descent, for comparison with §1.4

Nothing stops us from walking downhill on $\mathscr{L}(W)$ instead of solving
it, which is what §1.4 must do once the loss stops being quadratic. Same
argmin, same optimum — only slower, and revealingly so: the Hessian of the
per-example loss is $\tfrac{1}{n}X^{\top}X = \tfrac{1}{n}\mathrm{diag}(c_j)$, so
each row of $W$ descends at a rate proportional to how often its character
appears. The rate is capped by the *most* frequent character while the *least*
frequent one crawls, which is ill-conditioning ($\kappa = c_{\max}/c_{\min}$)
felt as wall-clock.

In [ ]:
print("\n\n--- TRAINING (gradient descent, same argmin) ---")
g = torch.Generator().manual_seed(1337+1)
W_gd_VV = torch.randn((V, V), generator=g, requires_grad=True)
K = 1000
lr = 4.0
print(f'{D=}, {N=}, {K=}, {lr=}')
for k in range(K):
  Y_NV = X_NV @ W_gd_VV                                    # forward: no exp, no normalize
  loss_gd = 0.5*((Y_NV - T_NV)**2).sum(dim=1).mean()       # squared error, not negative log likelihood
  if k % 100 == 0 or k == K-1: print(f'{k=}, loss={loss_gd.item()}')

  W_gd_VV.grad = None
  loss_gd.backward()                                        # grad = (1/N) X^T (XW - T)
  W_gd_VV.data += -lr*W_gd_VV.grad

diff_VV = (W_gd_VV.data - W_VV).abs()
slowest = diff_VV.max(dim=1).values.argmax().item()
print(f'\nclosed form loss: {loss}')
print(f'||W_gd - W_hat||_max: {diff_VV.max():.2e}')
print(f'slowest row to converge: {i2c[slowest]!r}, seen {int(counts_V[slowest])} times (rarest input character)')

### Inference: there is none, and that is the point

$\mathbf{y}(\mathbf{x})$ is a vector of reals, not a distribution. Here it
*happens* to look like one — because on one-hot inputs the fitted values are
class-conditional means, i.e. relative frequencies, so they land in $[0,1]$ and
sum to one. The summing to one is not luck and not specific to this dataset:
whenever the targets are 1-of-K coded and the constant function lies in the span
of the inputs, the components of $\mathbf{y}(\mathbf{x})$ sum to one for *every*
$\mathbf{x}$ (Bishop §5.1.4). Staying inside $[0,1]$ is the accident, and the
toy below breaks it with a single distant point, which also drags the decision
boundary across a correctly-classified example. Squared error punishes
predictions for being *too far on the correct side*, since a target of 1 is a
target and not a floor — which is the defect logistic regression fixes in §1.4
by putting a softmax between the weighted sum and the loss.

In [ ]:
print("\n\n--- INFERENCE ---")
Y_NV = f(X_NV, W_VV)
print(f'sum_k y_k(x) in [{Y_NV.sum(dim=1).min():.6f}, {Y_NV.sum(dim=1).max():.6f}] (1-of-K coding forces this)')
print(f'y_k(x)         in [{Y_NV.min():.6f}, {Y_NV.max():.6f}]')

# a 1-d two-class toy, 1-of-K coded, with the bias augmentation Bishop uses when the inputs are not one-hot
def lsq_fit(x_N, t_N, K=2):
  Xtilde_N2 = torch.stack([torch.ones_like(x_N), x_N], dim=1) # x~ = (1, x)^T
  T_NK = F.one_hot(t_N, num_classes=K).float()
  Wtilde_2K = torch.linalg.lstsq(Xtilde_N2, T_NK).solution
  return Xtilde_N2 @ Wtilde_2K

x_N = torch.tensor([0., 1., 2., 3., 4., 5.]); t_N = torch.tensor([0, 0, 0, 1, 1, 1])
Y_clean_NK = lsq_fit(x_N, t_N)
print(f'\nclean: y(x) =\n{Y_clean_NK}')
print(f'clean decisions: {Y_clean_NK.argmax(dim=1).tolist()} vs targets {t_N.tolist()}')

x_out_N = torch.cat([x_N, torch.tensor([30.])]); t_out_N = torch.cat([t_N, torch.tensor([1])])
Y_out_NK = lsq_fit(x_out_N, t_out_N)
print(f'\none distant point added: y(x) =\n{Y_out_NK}')
print(f'outlier decisions: {Y_out_NK.argmax(dim=1).tolist()} vs targets {t_out_N.tolist()} <- x=3 now misclassified')
print(f'rows still sum to one: {Y_out_NK.sum(dim=1).tolist()}, but y_k left [0,1]: min {Y_out_NK.min():.3f}, max {Y_out_NK.max():.3f}')

In [ ]:
print("\n\n--- TEST ---")
yhat_N = decide(Y_NV)                                           # argmax_k y_k(x), the one-step decision
acc = (yhat_N == yindices_N).float().mean()
print(f'squared error loss: {loss}')
print(f'accuracy: {acc:.4f} (the bigram argmax: one prediction per input character, so this is the ceiling for V=27 discriminants)')

for i in range(5):                                              # the first word's (x^(i), y^(i)) pairs
  xchar, ychar = i2c[xindices_N[i].item()], i2c[yindices_N[i].item()]
  print(f'(x^({i}),y^({i})): ({xchar},{ychar}) ---> y(x^({i}))[{ychar}]: {Y_NV[i, yindices_N[i]]:.4f}, decision: {i2c[yhat_N[i].item()]!r}')

# no average negative log likelihood to report, because y(x) is not a distribution:
# -log y_k(x) is undefined the moment a component is <= 0, and comparing against 1.4's
# avgnll would be comparing a score to a non-score. that is what 1.4 buys with inference.
print(f'\ncharacters the discriminants ever decide: {sorted({i2c[j.item()] for j in yhat_N.unique()})}')
print('every input character maps to exactly one output character, so "generation" here is a deterministic walk:')
for start in ['.', 'a', 'j']:
  out, ch = [start], start
  for _ in range(10):
    ch = i2c[decide(f(F.one_hot(torch.tensor([c2i[ch]]), num_classes=V).float(), W_VV))[0].item()]
    out.append(ch)
    if ch == '.': break
  print(f'  {"".join(out)}')  # collapses immediately: a decision cannot be sampled